=============================================================================
**PHASE 3 — NOTEBOOK 6: ARCHITECTURE DIAGRAM**  
=============================================================================
**RUN AFTER: 05_ablation_study.py**  

**WHAT THIS NOTEBOOK DOES:**  
  Generates a publication-quality architecture diagram showing the
  complete Phase 3 hybrid pipeline.

  The rubric specifically requires:
    "Professional-grade. Uses standard notation. Visualises the specific
     fusion mechanism (e.g., concatenation vs multiplication).
     Aesthetically suitable for a conference paper."

  This diagram shows:
    - Stream 1: DL pipeline (EfficientNet-B3 → Head → Dual outputs)
    - Stream 2: ML pipeline (Image → Hand-crafted features → GP)
    - Fusion: GP uncertainty signal feeds OOD decision
    - Wrapper: Conformal Prediction with formal coverage guarantee
    - All uncertainty signals explicitly labelled
    - Data dimensions at each stage

**ESTIMATED TIME: ~2 minutes (pure matplotlib)**  

**SAVES:**  
  outputs/architecture_diagram.png
  outputs/architecture_diagram_simple.png  (for slides)
=============================================================================

In [2]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patheffects as pe

OUTPUTS_DIR = "/teamspace/studios/this_studio/outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# ── COLOURS ───────────────────────────────────────────────────────────
NAVY   = '#0A1628'
DEEP   = '#0D3B66'
TEAL   = '#0891B2'
GREEN  = '#059669'
AMBER  = '#D97706'
RED    = '#DC2626'
SLATE  = '#334155'
MUTED  = '#64748B'
WHITE  = '#FFFFFF'
LIGHT  = '#F0F9FF'
LGREEN = '#F0FDF4'
LGRAY  = '#F8FAFC'


def rounded_box(ax, x, y, w, h, color, text_lines,
                 fontsize=8, text_color=WHITE,
                 alpha=1.0, radius=0.02):
    """Draw a rounded rectangle with centred multi-line text."""
    box = FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle=f"round,pad=0",
        facecolor=color, edgecolor=WHITE,
        linewidth=1.2, alpha=alpha, zorder=3
    )
    ax.add_patch(box)
    # Multi-line text
    full_text = '\n'.join(text_lines)
    ax.text(x, y, full_text,
            ha='center', va='center', fontsize=fontsize,
            color=text_color, fontweight='bold',
            multialignment='center', zorder=4,
            linespacing=1.4)


def arrow(ax, x1, y1, x2, y2, color=SLATE, lw=1.5,
           label=None, label_side='right'):
    """Draw a clean arrow between two points."""
    ax.annotate(
        '', xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle='->', color=color,
            lw=lw, connectionstyle='arc3,rad=0.0'
        ),
        zorder=2
    )
    if label:
        mx = (x1 + x2) / 2
        my = (y1 + y2) / 2
        dx = 0.03 if label_side == 'right' else -0.03
        ax.text(mx + dx, my, label,
                fontsize=6.5, color=color,
                ha='left' if label_side == 'right' else 'right',
                va='center', style='italic', zorder=5)


def section_label(ax, x, y, text, color):
    ax.text(x, y, text, fontsize=8, color=color,
            fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor=color, edgecolor=color, alpha=0.15),
            zorder=6)

In [3]:
# FULL ARCHITECTURE DIAGRAM

In [4]:
fig, ax = plt.subplots(figsize=(18, 11))
ax.set_xlim(0, 18)
ax.set_ylim(0, 11)
ax.set_aspect('equal')
ax.axis('off')
fig.patch.set_facecolor(LGRAY)
ax.set_facecolor(LGRAY)

# Title
ax.text(9, 10.6,
        'Phase 3 Hybrid Architecture — Robust Medical Vision',
        fontsize=14, fontweight='bold', color=NAVY,
        ha='center', va='center')
ax.text(9, 10.25,
        'EfficientNet-B3 + MC Dropout + Evidential Head + '
        'Temperature Scaling + Mahalanobis OOD + '
        'GP Signal + Conformal Prediction',
        fontsize=8, color=MUTED, ha='center', va='center')

# ── Dividing line between DL and ML streams ────────────────────────
ax.axhline(y=5.5, xmin=0.01, xmax=0.99,
            color=MUTED, linewidth=0.5, linestyle='--', alpha=0.4)
ax.text(0.3, 5.5, 'Stream 1\n(DL)',
        fontsize=8, color=TEAL, fontweight='bold',
        ha='center', va='center')
ax.text(0.3, 3.0, 'Stream 2\n(ML)',
        fontsize=8, color=AMBER, fontweight='bold',
        ha='center', va='center')

# ══════════════════════════════════════════
# INPUT
# ══════════════════════════════════════════
rounded_box(ax, 1.1, 7.0, 1.6, 0.75, NAVY,
             ['Input Image', '224×224×3'], fontsize=8)

# Arrow from input to backbone
arrow(ax, 1.9, 7.0, 2.6, 7.0, TEAL, lw=2)

# ══════════════════════════════════════════
# STREAM 1 — DL PIPELINE
# ══════════════════════════════════════════

# EfficientNet-B3 Backbone
rounded_box(ax, 3.5, 7.0, 1.7, 0.85, DEEP,
             ['EfficientNet-B3', 'ImageNet Pretrained',
              '1536-dim features'], fontsize=7.5)

# Annotation: two-stage training
ax.text(3.5, 7.6, '2-stage training\n(freeze → unfreeze)',
        fontsize=6, color=MUTED, ha='center', style='italic')

arrow(ax, 4.35, 7.0, 5.1, 7.0, TEAL, lw=2,
       label='1536-dim')

# Global Average Pooling
rounded_box(ax, 5.55, 7.0, 0.8, 0.55, SLATE,
             ['GAP', '→ flat'], fontsize=7)
arrow(ax, 5.95, 7.0, 6.5, 7.0, TEAL, lw=1.5)

# Custom Head
rounded_box(ax, 7.2, 7.0, 1.3, 1.5, DEEP,
             ['Custom Head',
              'Dense(1536→512)',
              'BatchNorm + GELU',
              'MCDropout(p=0.4)',
              'Dense(512→256)',
              'GELU',
              'MCDropout(p=0.3)'],
             fontsize=6.5)

# MCDropout annotation
ax.annotate(
    '20 passes at inference\n→ epistemic uncertainty',
    xy=(7.2, 6.55), xytext=(5.4, 5.9),
    fontsize=6, color=AMBER, style='italic',
    arrowprops=dict(arrowstyle='->', color=AMBER,
                    lw=0.8, connectionstyle='arc3,rad=0.2'),
    zorder=5
)

arrow(ax, 7.85, 7.0, 8.5, 7.0, TEAL, lw=1.5,
       label='256-dim')

# Fork point
fork_x, fork_y = 8.7, 7.0
ax.plot(fork_x, fork_y, 'o', color=TEAL, markersize=6, zorder=5)

# ── Standard Head ───────────────────────────────────────────
arrow(ax, fork_x, fork_y + 0.05, fork_x, 8.5, TEAL, lw=1.2)
arrow(ax, fork_x, 8.5, 9.5, 8.5, TEAL, lw=1.2)
rounded_box(ax, 10.3, 8.5, 1.5, 0.65, '#2563EB',
             ['Standard Head', '7 logits → softmax'], fontsize=7.5)

# ── Evidential Head ─────────────────────────────────────────
arrow(ax, fork_x, fork_y - 0.05, fork_x, 5.85, TEAL, lw=1.2)
arrow(ax, fork_x, 5.85, 9.5, 5.85, TEAL, lw=1.2)
rounded_box(ax, 10.3, 5.85, 1.5, 0.65, GREEN,
             ['Evidential Head', '7 Dirichlet α'], fontsize=7.5)

# Evidential annotation
ax.text(10.3, 5.3,
        'aleatoric + epistemic\nuncertainty (1 pass)',
        fontsize=6, color=GREEN, ha='center', style='italic')

# Temperature Scaling
arrow(ax, 11.05, 8.5, 11.8, 8.5, TEAL, lw=1.2)
rounded_box(ax, 12.5, 8.5, 1.3, 0.65, '#7C3AED',
             ['Temperature', 'Scaling  T=?',
              'fixes ECE ~0.06'], fontsize=7)

# Mahalanobis OOD
arrow(ax, fork_x, fork_y, fork_x + 0.3, 7.0, TEAL, lw=0.5)
ax.annotate(
    '', xy=(12.5, 7.5), xytext=(8.8, 7.0),
    arrowprops=dict(arrowstyle='->', color=AMBER, lw=1.2,
                    connectionstyle='arc3,rad=-0.25'),
    zorder=2
)
rounded_box(ax, 12.5, 7.5, 1.4, 0.65, AMBER,
             ['Mahalanobis', 'OOD Detector',
              'on 256-dim feats'], fontsize=7, text_color=WHITE)

# ══════════════════════════════════════════
# STREAM 2 — ML PIPELINE
# ══════════════════════════════════════════

# Same input image
rounded_box(ax, 1.1, 3.0, 1.6, 0.75, NAVY,
             ['Input Image', '224×224×3'], fontsize=8)
arrow(ax, 1.9, 3.0, 2.6, 3.0, AMBER, lw=1.5)

# Feature extraction
rounded_box(ax, 3.6, 3.0, 1.8, 1.1, '#92400E',
             ['Feature Extraction',
              'GLCM (40-dim)',
              'LBP  (26-dim)',
              'HSV  (32-dim)',
              'RGB stats (6-dim)'],
             fontsize=6.5, text_color=WHITE)
ax.text(3.6, 2.35, '→ 104-dim vector',
        fontsize=6.5, color=AMBER, ha='center', style='italic')

arrow(ax, 4.5, 3.0, 5.2, 3.0, AMBER, lw=1.5,
       label='104-dim')

# Preprocessing
rounded_box(ax, 5.8, 3.0, 1.0, 0.7, SLATE,
             ['Scale', '+', 'PCA(50)'], fontsize=7)
ax.text(5.8, 2.55, '50-dim',
        fontsize=6.5, color=MUTED, ha='center', style='italic')

arrow(ax, 6.3, 3.0, 7.0, 3.0, AMBER, lw=1.5)

# Gaussian Process
rounded_box(ax, 7.9, 3.0, 1.5, 0.8, '#B45309',
             ['Gaussian Process', 'Classifier',
              'one-vs-rest'], fontsize=7, text_color=WHITE)

arrow(ax, 8.65, 3.0, 9.4, 3.0, AMBER, lw=1.5)

# Isolation Forest
rounded_box(ax, 10.25, 3.0, 1.4, 0.7, '#D97706',
             ['Isolation Forest', 'OOD Detection'],
             fontsize=7.5, text_color=WHITE)

# GP uncertainty output
ax.text(8.65, 2.5,
        'GP probabilities\n+ uncertainty (entropy)',
        fontsize=6, color=AMBER, ha='center', style='italic')

# ══════════════════════════════════════════
# FUSION — HYBRID OOD DECISION
# ══════════════════════════════════════════

# Fusion box
rounded_box(ax, 13.6, 5.5, 1.6, 0.8, '#1E3A5F',
             ['Hybrid OOD', 'Decision',
              'Mahal. OR IsoForest'], fontsize=7)

# Arrows from Mahalanobis and IsoForest to fusion
arrow(ax, 13.2, 7.5, 13.5, 5.9, AMBER, lw=1.2)
arrow(ax, 11.0, 3.0, 13.5, 5.1, AMBER, lw=1.2)

ax.text(12.4, 4.6, 'UNION\n(A OR B)',
        fontsize=7, color='#1E3A5F', fontweight='bold',
        ha='center', style='italic')

# Calibrated probs from temperature scaling to conformal
arrow(ax, 13.15, 8.5, 14.7, 8.5, '#7C3AED', lw=1.2)

# ══════════════════════════════════════════
# CONFORMAL PREDICTION
# ══════════════════════════════════════════
rounded_box(ax, 15.5, 8.5, 1.6, 0.9, '#5B21B6',
             ['Conformal', 'Prediction', '95% coverage'], fontsize=7.5)

# ══════════════════════════════════════════
# FINAL OUTPUTS
# ══════════════════════════════════════════

# Output collector
rounded_box(ax, 15.5, 6.5, 1.8, 2.5, NAVY,
             ['FINAL OUTPUT',
              '',
              'Prediction set',
              '{mel, nv}',
              '',
              'MC uncertainty',
              'Ev. aleatoric',
              'Ev. epistemic',
              '',
              'OOD flag',
              'CP coverage'],
             fontsize=6.5)

# Arrows to output
arrow(ax, 15.5, 8.05, 15.5, 7.75, '#5B21B6', lw=1.2)
arrow(ax, 13.6, 5.1, 14.6, 6.8, '#1E3A5F', lw=1.2)
arrow(ax, 11.05, 5.85, 14.6, 6.3, GREEN, lw=1.0)

# ══════════════════════════════════════════
# CLINICAL DECISION GATEWAY
# ══════════════════════════════════════════
rounded_box(ax, 15.5, 4.5, 1.8, 1.3, '#065F46',
             ['Clinical Gate',
              'OOD? → Reject',
              'Large set? → Refer',
              'Size=1? → Classify'],
             fontsize=6.5)
arrow(ax, 15.5, 5.75, 15.5, 5.15, NAVY, lw=1.5)

# ══════════════════════════════════════════
# LEGEND
# ══════════════════════════════════════════
legend_items = [
    (DEEP,   'DL Feature Extraction'),
    (AMBER,  'ML / Hand-crafted Pipeline'),
    ('#2563EB', 'Classification Head'),
    (GREEN,  'Evidential Uncertainty'),
    ('#7C3AED', 'Conformal Prediction'),
    ('#1E3A5F', 'Hybrid Fusion'),
    ('#065F46', 'Clinical Safety Gate'),
]
lx, ly = 0.4, 1.8
ax.text(lx, ly + 0.3, 'Legend:', fontsize=8,
        fontweight='bold', color=NAVY)
for i, (col, label) in enumerate(legend_items):
    yl = ly - i * 0.28
    ax.add_patch(plt.Rectangle(
        (lx - 0.15, yl - 0.09), 0.28, 0.18,
        facecolor=col, edgecolor=WHITE, linewidth=0.8, zorder=4
    ))
    ax.text(lx + 0.22, yl, label, fontsize=6.5,
            color=SLATE, va='center')

# Dimension labels at key points
for (x, y, text) in [
    (8.7, 7.35, '256-dim\nfeatures'),
    (12.5, 9.0, 'calibrated\nprobs'),
    (13.6, 6.2, 'OOD\nflag'),
]:
    ax.text(x, y, text, fontsize=6, color=MUTED,
            ha='center', style='italic')

plt.tight_layout()
diag_path = f"{OUTPUTS_DIR}/architecture_diagram.png"
plt.savefig(diag_path, dpi=180, bbox_inches='tight',
            facecolor=LGRAY)
plt.close()
print(f"Architecture diagram saved: {diag_path}")

Architecture diagram saved: /teamspace/studios/this_studio/outputs/architecture_diagram.png


In [5]:
# SIMPLIFIED VERSION FOR SLIDES

In [6]:
fig2, ax2 = plt.subplots(figsize=(14, 6))
ax2.set_xlim(0, 14)
ax2.set_ylim(0, 6)
ax2.axis('off')
fig2.patch.set_facecolor(NAVY)
ax2.set_facecolor(NAVY)

ax2.text(7, 5.65, 'Robust Medical Vision — Phase 3 Architecture',
          fontsize=13, fontweight='bold', color=WHITE,
          ha='center', va='center')

# Simplified 5-box pipeline
boxes = [
    (1.4,  2.8, 1.8, DEEP,    ['Input',  '224×224×3']),
    (3.8,  3.8, 1.8, DEEP,    ['EfficientNet-B3', '+ Custom Head', '256-dim']),
    (3.8,  1.8, 1.8, '#92400E', ['GLCM+LBP+HSV', '→ PCA(50)', '→ GP']),
    (7.0,  2.8, 2.0, '#1E3A5F', ['Uncertainty Pipeline',
                                   'MC Dropout ×30',
                                   'Evidential Head',
                                   'TempScale+Mahal.']),
    (10.2, 2.8, 2.0, '#5B21B6', ['Conformal', 'Prediction',
                                   '95% Coverage']),
    (12.8, 2.8, 1.8, '#065F46', ['Clinical', 'Output +', 'Safety Flag']),
]

for (x, y, w, col, lines) in boxes:
    rounded_box(ax2, x, y, w, 0.9 + len(lines)*0.1, col, lines,
                 fontsize=8)

# Arrows
arrow_coords = [
    (2.3, 3.8, 2.9, 3.8, TEAL),
    (2.3, 1.8, 2.9, 1.8, AMBER),
    (4.7, 3.8, 6.0, 3.1, TEAL),
    (4.7, 1.8, 6.0, 2.5, AMBER),
    (8.0, 2.8, 9.2, 2.8, WHITE),
    (11.2, 2.8, 11.9, 2.8, WHITE),
]
for (x1, y1, x2, y2, col) in arrow_coords:
    ax2.annotate('', xy=(x2, y2), xytext=(x1, y1),
                  arrowprops=dict(arrowstyle='->',
                                  color=col, lw=1.8),
                  zorder=5)

# Labels
ax2.text(1.4, 1.1, 'STREAM 1: Deep Learning',
          fontsize=7.5, color=TEAL, ha='center',
          fontweight='bold', style='italic')
ax2.text(1.4, 0.7, 'STREAM 2: Classical ML',
          fontsize=7.5, color=AMBER, ha='center',
          fontweight='bold', style='italic')
ax2.text(7.0, 0.5, 'FUSION',
          fontsize=7.5, color='#A8C4D4', ha='center',
          fontweight='bold', style='italic')

plt.tight_layout()
simple_path = f"{OUTPUTS_DIR}/architecture_diagram_simple.png"
plt.savefig(simple_path, dpi=180, bbox_inches='tight',
            facecolor=NAVY)
plt.close()
print(f"Simplified diagram saved: {simple_path}")

print(f"""
Architecture diagrams generated:
  {OUTPUTS_DIR}/architecture_diagram.png        (full, for report)
  {OUTPUTS_DIR}/architecture_diagram_simple.png (for slides)

→ NEXT: Run 07_generate_report_data.py
""")

Simplified diagram saved: /teamspace/studios/this_studio/outputs/architecture_diagram_simple.png

Architecture diagrams generated:
  /teamspace/studios/this_studio/outputs/architecture_diagram.png        (full, for report)
  /teamspace/studios/this_studio/outputs/architecture_diagram_simple.png (for slides)

→ NEXT: Run 07_generate_report_data.py

